In [ ]:
import yt_dlp as youtube_dl
import os

def download_audio(url, output_directory, ffmpeg_path):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': os.path.join(output_directory, '%(title)s.%(ext)s'),  # Keep the original template
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'ffmpeg_location': ffmpeg_path,  # Specify the path to ffmpeg and ffprobe
    }

    with youtube_dl.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(url, download=True)
        original_path = ydl.prepare_filename(info_dict)
        
        # Determine the WAV file path
        base, _ = os.path.splitext(original_path)
        wav_path = base + '.wav'
        
        # Remove spaces from the WAV file path
        new_wav_path = wav_path.replace(' ', '')
        
        # Rename the WAV file if it exists
        if os.path.exists(wav_path):
            os.rename(wav_path, new_wav_path)
            print(f'Renamed file to: {new_wav_path}')
        else:
            print(f'File {wav_path} does not exist.')

url = 'https://www.youtube.com/watch?v=OUvlamJN3nM'
output_directory = '/mnt/c/Users/dakot/Music/wav_files'
ffmpeg_path = '/usr/bin/ffmpeg'
download_audio(url, output_directory, ffmpeg_path)


In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100  # Set the limit to 100 MB

In [ ]:
# Step 1: Enable inline plotting in the notebook
%matplotlib inline

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.fft import fft
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

def analyze_audio(file_path, interval_ms=10):
    # Read the audio file
    sample_rate, audio_data = wavfile.read(file_path)
    
    # Ensure audio data is mono
    if len(audio_data.shape) == 2:
        audio_data = audio_data.mean(axis=1)
    
    # Calculate the number of samples per interval
    interval_samples = int(sample_rate * interval_ms / 1000)
    
    # List to hold analysis results
    results = []

    for start in range(0, len(audio_data), interval_samples):
        end = start + interval_samples
        segment = audio_data[start:end]

        if len(segment) == 0:
            continue

        # Perform Fourier transform
        yf = fft(segment)
        xf = np.fft.fftfreq(len(segment), 1 / sample_rate)
        
        # Get the magnitude
        magnitudes = np.abs(yf)
        
        # Get the dominant frequency and its amplitude
        dominant_index = np.argmax(magnitudes[:len(magnitudes)//2])  # Ignore negative frequencies
        dominant_frequency = xf[dominant_index]
        dominant_amplitude = magnitudes[dominant_index]
        
        # Append the results
        results.append({
            'time': start / sample_rate,
            'frequency': abs(dominant_frequency),
            'amplitude': dominant_amplitude
        })

    return pd.DataFrame(results)

def solve_wave_equation(frequency, amplitude, grid_size=(250, 250), c=1):
    x = np.linspace(0, 1, grid_size[0])
    y = np.linspace(0, 1, grid_size[1])
    X, Y = np.meshgrid(x, y)
    Z = amplitude * np.sin(2 * np.pi * frequency * X) * np.sin(2 * np.pi * frequency * Y)
    return X, Y, Z

def update_frame(frame, data, ax):
    ax.clear()
    frequency = data.iloc[frame]['frequency']
    amplitude = data.iloc[frame]['amplitude']
    if frequency > 0:  # Only update if frequency is non-zero
        X, Y, Z = solve_wave_equation(frequency, amplitude)
        contour_set = ax.contour(X, Y, Z, levels=[0], colors='black')
        ax.set_title(f'Time: {data.iloc[frame]["time"]:.2f}s, Frequency: {frequency:.2f}Hz')
        return contour_set.collections
    return []

def main(file_path):
    data = analyze_audio(file_path)
    data = data[data['frequency'] > 0].reset_index(drop=True)  # Filter out zero frequencies

    fig, ax = plt.subplots()

    def init():
        return []

    ani = animation.FuncAnimation(
        fig, update_frame, frames=len(data),
        fargs=(data, ax), interval=100, blit=True, init_func=init
    )

    return ani

# Example usage
#file_path = "/mnt/c/Users/dakot/Music/wav_files/440Hz(10secondsofA).wav"
file_path = "/mnt/c/Users/dakot/Music/wav_files/LiesofPOST-FEEL(LPVer.).wav"
ani = main(file_path)
HTML(ani.to_jshtml())
